# 01 — Build life tables from CONAPO data

Implements [METHODOLOGY.md §1](../docs/methodology.md). Reads CONAPO's projected deaths and mid-year population by single age, sex and state for one reference year, converts them to central death rates and then to a complete life table (`qx, lx, dx, Lx, Tx, ex`) for the 32 states plus the national aggregate.

Output: `data/processed/life_tables.csv`.

Why deaths ÷ population instead of a published `qx`: CONAPO's open-data release does not include `qx` directly, only the deaths and populations its own projection model produced. Dividing the two recovers the mortality schedule CONAPO used, which is still "CONAPO's mortality", not an independent estimate from raw registrations. See [ADR 0004](../docs/0004_qx-from-conapo-deaths-and-population.md).

In [1]:
import pandas as pd

from vitaemx_research import conapo, lifetable

REFERENCE_YEAR = 2023
PROCESSED = conapo.RAW_DIR.parent / "processed"
PROCESSED.mkdir(exist_ok=True)

## 1. Load deaths and exposure

One row per (state, sex, age). `sex = total` and `state_code = 0` (national) are aggregates built by summing the CONAPO rows.

In [2]:
exposure = conapo.load_deaths_and_population(REFERENCE_YEAR)
print(exposure.shape)
exposure.head()

(10890, 6)


,state_code,state_name,sex,age,deaths,population
0,0,República Mexicana,female,0,10706,1004586
1,0,República Mexicana,female,1,1369,1013987
2,0,República Mexicana,female,2,575,1021585
3,0,República Mexicana,female,3,369,1024195
4,0,República Mexicana,female,4,283,1027301


In [3]:
exposure.groupby("sex")[["deaths", "population"]].sum()

,deaths,population
sex,,
female,701146,133994368
male,909480,128276306
total,1610626,262270674


## 2. Build the tables

`mx = deaths / population`, `qx = mx / (1 + (1 - ax)·mx)` with `ax = 0.5` (`0.1` at age 0), then the standard recursion for `lx`, `dx`, `Lx`, `Tx`, `ex`. The last age with any exposure is treated as an open interval (`qx = 1`).

In [4]:
life_tables = lifetable.build_all_life_tables(exposure)
national = life_tables[(life_tables.state_code == 0) & (life_tables.sex == "total")]
national[national.age.isin([0, 1, 5, 15, 30, 45, 60, 75, 90, 100, 109])].round(6)

,state_code,state_name,sex,age,mx,qx,lx,dx,Lx,Tx,ex
220,0,República Mexicana,total,0,0.011890,0.011764,100000.000000,1176.394275,98941.245153,7.550094e+06,75.500944
221,0,República Mexicana,total,1,0.001464,0.001463,98823.605725,144.539975,98751.335738,7.451153e+06,75.398515
225,0,República Mexicana,total,5,0.000225,0.000225,98561.613234,22.201719,98550.512375,7.056571e+06,71.595534
235,0,República Mexicana,total,15,0.000621,0.000621,98292.926982,61.059696,98262.397134,6.072067e+06,61.775220
250,0,República Mexicana,total,30,0.002843,0.002839,95715.264620,271.692760,95579.418240,4.612717e+06,48.192071
265,0,República Mexicana,total,45,0.003746,0.003739,91235.687058,341.164547,91065.104784,3.209571e+06,35.178904
280,0,República Mexicana,total,60,0.009905,0.009856,83576.608198,823.717647,83164.749375,1.890058e+06,22.614676
295,0,República Mexicana,total,75,0.034988,0.034387,62892.259337,2162.649273,61810.934700,7.668186e+05,12.192575
310,0,República Mexicana,total,90,0.132405,0.124184,22026.926361,2735.394284,20659.229219,1.147766e+05,5.210743
320,0,República Mexicana,total,100,0.330450,0.283594,2776.152042,787.298913,2382.502586,7.176749e+03,2.585142


## 3. Sanity checks

- survivors never increase with age,
- deaths add up to the radix,
- life expectancy at birth lands in a plausible range for Mexico (low 70s for men, high 70s for women).

In [5]:
assert (life_tables.groupby(["state_code", "sex"])["lx"].diff().dropna() <= 0).all()
assert ((life_tables.groupby(["state_code", "sex"])["dx"].sum() - lifetable.RADIX).abs() < 1e-6).all()

e0 = life_tables[life_tables.age == 0].pivot(index="state_name", columns="sex", values="ex").round(2)
assert e0["male"].between(65, 80).all() and e0["female"].between(70, 85).all()
e0.sort_values("total")

sex,female,male,total
state_name,,,
Chiapas,76.94,70.00,73.48
Guerrero,77.10,70.14,73.69
Oaxaca,77.28,70.36,73.90
Tabasco,77.55,70.83,74.18
Veracruz,77.58,70.80,74.24
Puebla,77.69,70.97,74.40
Hidalgo,77.71,71.01,74.42
Michoacán,77.96,71.26,74.61
Tlaxcala,77.92,71.24,74.62


## 4. Write output

In [6]:
out = PROCESSED / "life_tables.csv"
life_tables.to_csv(out, index=False, float_format="%.10g")
print(out, life_tables.shape)

/home/user/vitaemx/data/processed/life_tables.csv (10890, 11)
